# 06 - Lakehouse y Transacciones ACID con Delta Lake 3.x

### Capacidades de Delta Lake
* **Transacciones ACID**: Previene datos corruptos gracias al registro `_delta_log`.
* **Mutaciones**: Permite `UPDATE` y `DELETE` sin reescribir tablas completas.
* **Time Travel**: Permite consultar instantáneas históricas exactas de la tabla.


In [ ]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA
from delta.tables import DeltaTable
import pyspark.sql.functions as F

spark = get_spark_session("06_Delta")
dir_delta = "../data/delta/deportistas_lake"
df_dep = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")

df_dep.write.format("delta").mode("overwrite").partitionBy("genero").save(dir_delta)
tbl = DeltaTable.forPath(spark, dir_delta)
tbl.update(condition="deportista_id = 1", set={"nombre": F.lit("A Dijiang (Updated)")})

print("Time Travel - Version 0 inicial:")
spark.read.format("delta").option("versionAsOf", 0).load(dir_delta).filter("deportista_id = 1").select("deportista_id", "nombre").show()
print("Version 1 actualizada:")
spark.read.format("delta").load(dir_delta).filter("deportista_id = 1").select("deportista_id", "nombre").show()
